| Feature          | FAISS                          | Chroma                             | Pinecone                                             |
| ---------------- | ------------------------------ | ---------------------------------- | ---------------------------------------------------- |
| Product type     | Vector-search library          | Vector database                    | Managed vector database                              |
| Runs locally     | Yes                            | Yes                                | No normal local database mode                        |
| Managed cloud    | No                             | Yes                                | Yes, primary mode                                    |
| Vector index     | Directly controlled            | Database-managed                   | Fully managed                                        |
| Documents        | LangChain/external store       | Native collection record           | Metadata/document fields through records/integration |
| Metadata         | External/wrapper               | Native                             | Native                                               |
| Metadata filters | Not native FAISS DB filtering  | Native                             | Native                                               |
| CRUD             | Limited/index-dependent        | Native                             | Native                                               |
| Persistence      | Manual save/load               | Automatic persistent client/server | Managed                                              |
| Collections      | No native collection primitive | Native collections                 | Indexes and namespaces                               |
| Scaling          | You design it                  | Local/server/cloud options         | Automatically managed infrastructure                 |
| Server API       | Build yourself                 | Available                          | Built-in                                             |
| GPU/index tuning | Strong direct control          | Abstracted                         | Fully abstracted                                     |
| Cost             | Infrastructure only            | Local free/cloud paid              | Cloud usage-based                                    |
| Best for         | Research/local/custom ANN      | Local RAG and flexible deployments | Production cloud-scale retrieval                     |


In [4]:
# pip install pinecone

In [5]:
import os
import time
from uuid import uuid4
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document

In [6]:
# ---------------------------------------------------
# 1. Load environment variables
# ---------------------------------------------------

load_dotenv()

google_api_key = os.getenv("OPENAI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

if not google_api_key:
    raise ValueError("OPENAI_API_KEY is missing")

if not pinecone_api_key:
    raise ValueError("PINECONE_API_KEY is missing")

In [7]:
# ---------------------------------------------------
# 2. Embedding model
# ---------------------------------------------------
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings(model="text-embedding-3-large")
dimension = len(
    embeddings.embed_query("dimension check")
)
print("Embedding dimension:", dimension)


Embedding dimension: 3072


In [8]:
# ---------------------------------------------------
# 3. Pinecone client
# ---------------------------------------------------
pc = Pinecone(
    api_key=pinecone_api_key
)

In [9]:
index_name = "langchain-llama-index"

In [10]:
# ---------------------------------------------------
# 4. Create index
# ---------------------------------------------------
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
    )

In [11]:
description = pc.describe_index(index_name)
description.status["ready"]

True

In [12]:
# ---------------------------------------------------
# 5. Wait for index readiness
# ---------------------------------------------------
while True:
    description = pc.describe_index(index_name)
    if description.status["ready"]:
        break
    time.sleep(2)

In [13]:
# ---------------------------------------------------
# 6. Connect to index
# ---------------------------------------------------
index = pc.Index(index_name)
print("Index connected:", index_name)

Index connected: langchain-llama-index


In [14]:
vector_store=PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace="demo-documents"
)

In [15]:
# ---------------------------------------------------
# 8. Create documents
# ---------------------------------------------------

documents = [
    Document(
        page_content=(
            "I had chocolate chip pancakes and "
            "scrambled eggs for breakfast this morning."
        ),
        metadata={"source": "tweet"},
    ),
    Document(
        page_content=(
            "The weather forecast for tomorrow is cloudy "
            "and overcast, with a high of 62 degrees."
        ),
        metadata={"source": "news"},
    ),
    Document(
        page_content=(
            "Building an exciting new project with "
            "LangChain - come check it out!"
        ),
        metadata={"source": "tweet"},
    ),
    Document(
        page_content=(
            "Robbers broke into the city bank and "
            "stole $1 million in cash."
        ),
        metadata={"source": "news"},
    ),
    Document(
        page_content=(
            "LangGraph is the best framework for building "
            "stateful, agentic applications!"
        ),
        metadata={"source": "tweet"},
    ),
]

ids = [
    str(uuid4())
    for _ in documents
]




In [16]:
# ---------------------------------------------------
# 9. Add documents
# ---------------------------------------------------

inserted_ids = vector_store.add_documents(
    documents=documents,
    ids=ids
)

print("Inserted IDs:", inserted_ids)

Inserted IDs: ['7613a9cf-bd19-4793-84ef-34957c05e76b', '8c01fd25-4f48-492b-af29-6a1f20a3a4af', '1d659bb8-94a4-4130-bb20-4aa44b96d182', '9ebe74fa-ce36-4868-b2db-d31a9e71ea53', 'add6caca-b18a-44dd-b32b-58778870e91b']


Inserted IDs: [
'd2d7db20-2306-4cf6-be4e-a95eea1bf814', 
'540d2ceb-c522-48ec-91fe-a25e7df13882', 
'e9841e9b-3897-444d-b6ba-9226bd3ed238',
 '3b726a4f-59c0-4ff7-9d24-c29c624ba3c3', 
 'cc067870-168e-4cf0-8588-fa3985cf4944']

In [17]:
# ---------------------------------------------------
# 10. Similarity search with metadata filter
# ---------------------------------------------------

results = vector_store.similarity_search(
    query=(
        "LangChain provides abstractions "
        "for working with LLMs"
    ),
    k=2,
    filter={
        "source": "tweet"
    }
)

for result in results:
    print(result.page_content)
    print(result.metadata)





Building an exciting new project with LangChain - come check it out!
{'source': 'tweet'}
LangGraph is the best framework for building stateful, agentic applications!
{'source': 'tweet'}


In [18]:

# ---------------------------------------------------
# 11. Retriever with threshold and filter
# ---------------------------------------------------

retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 1,
        "score_threshold": 0.4,
        "filter": {
            "source": "news"
        }
    }
)

retrieved_docs = retriever.invoke(
    "Stealing money from a bank is a crime"
)

print(retrieved_docs)

[Document(id='9ebe74fa-ce36-4868-b2db-d31a9e71ea53', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]
